# Find the recordings worth investigating

**A hands-on robotics data audit | CPU only | No robot or API key needed**

You have a batch of demonstrations. Before training, which recordings need a closer look, and what evidence supports that decision?

In this notebook you will **install → audit → explore** a small public synthetic LeRobot dataset. The fixture deliberately contains action spikes. The grader measures the files; the fixture name is not evidence that every injected event will be detected.

**You will leave with:** a measured score, episode decisions, a traceable findings table, and downloadable HTML/JSON reports.

Choose **Runtime → Run all** on a CPU runtime. The previously validated operational flow took about a minute, excluding runtime allocation; timing varies. This guided presentation adds no model calls. Packages and the public dataset are downloaded into your temporary Colab VM. No report is published to Kalanos.

> A high score does not establish training readiness. A review gate can fail while the notebook completes successfully. Missing evidence is not a pass.


## 1 · Install the grader
Run this cell once. Dependencies live in a separate environment, avoiding changes to the notebook kernel's analysis packages. `virtualenv` works around Colab's missing `ensurepip`. Kalanos and Hugging Face Hub are pinned; resolved dependencies are saved with your results.


In [ ]:
# @title Install the grader (click to show code)
import json
import os
import re
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

from IPython.display import HTML, FileLink, display


WORK = Path(tempfile.mkdtemp(prefix="kalanos-colab-"))
VENV = WORK / "venv"
OUTPUT = WORK / "results"
OUTPUT.mkdir()
# Colab's Python 3.13 lacks ensurepip, so `python -m venv` cannot seed pip.
# virtualenv bundles its own pip and works on Colab and local Python alike.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "-q", str(VENV)], check=True)
PYTHON = VENV / "bin" / "python"
KALANOS = VENV / "bin" / "kalanos"
subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        "kalanos[numeric,video]==0.10.0",
        "huggingface-hub==1.31.0",
    ],
    check=True,
)
freeze = subprocess.run(
    [str(PYTHON), "-m", "pip", "freeze"], capture_output=True, text=True
)
lock = freeze.stdout if freeze.returncode == 0 else "# pip freeze unavailable\n"
(OUTPUT / "requirements-resolved.txt").write_text(lock)
print("Working directory:", WORK)

## 2 · Choose and audit a dataset
Start with [synthetic-action-spikes](https://huggingface.co/datasets/kalanosai/synthetic-action-spikes): a small numeric-only teaching fixture, not real-robot training data. This example cannot demonstrate camera checks.

**Before running:** predict whether an aggregate score alone will identify every recording needing review.

The download is capped at 200 MB. A blank revision resolves `main` once, then downloads that immutable commit. The printed revision is saved in `provenance.json`. For a repeatable class exercise, set `DATASET_REVISION` to the previously validated commit `16cc33ab85094065744370740d17f3fce37698d3`.

If you change datasets, reset the revision and rerun from the start. The loader requires one unpacked LeRobot root and Parquet recordings; it will not silently grade only a README.


In [ ]:
# @title Download the dataset (click to show code)
DATASET_ID = "kalanosai/synthetic-action-spikes"
DATASET_REVISION = ""  # Set a validated 40-character Hugging Face commit SHA to replay.
MAX_BYTES = 200 * 1024 * 1024

if DATASET_REVISION and not re.fullmatch(r"[a-f0-9]{40}", DATASET_REVISION):
    raise ValueError(
        "Use an exact Hugging Face commit SHA, or leave blank to resolve once."
    )

download_script = r"""import json, pathlib, sys
from huggingface_hub import HfApi, snapshot_download
repo, revision, destination, max_bytes = sys.argv[1:]
api = HfApi(token=False)
info = api.dataset_info(repo, revision=revision or "main", files_metadata=True)
files = info.siblings
if not files or any(f.size is None for f in files):
    raise RuntimeError("Cannot verify repository size; no download performed.")
total = sum(f.size for f in files)
if total > int(max_bytes):
    raise RuntimeError(f"Repository is {total} bytes, above download limit.")
path = snapshot_download(repo_id=repo, repo_type="dataset", revision=info.sha,
                         local_dir=destination, token=False, max_workers=2)
print(json.dumps({"dataset_id":repo, "revision":info.sha,
                  "repository_bytes":total, "local_path":path}))
"""
env = dict(os.environ, HF_HUB_DISABLE_TELEMETRY="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
raw = subprocess.check_output(
    [
        str(PYTHON),
        "-c",
        download_script,
        DATASET_ID,
        DATASET_REVISION,
        str(WORK / "download"),
        str(MAX_BYTES),
    ],
    text=True,
    env=env,
)
provenance = json.loads(raw.strip().splitlines()[-1])
roots = [
    p.parent.parent for p in Path(provenance["local_path"]).rglob("meta/info.json")
]
if len(roots) != 1:
    raise RuntimeError(
        f"Expected one unpacked LeRobot root, found {len(roots)}. "
        "Inspect repository layout."
    )
DATASET = roots[0]
info = json.loads((DATASET / "meta/info.json").read_text())
if not list((DATASET / "data").rglob("*.parquet")):
    raise RuntimeError(
        "No LeRobot Parquet data found; do not grade metadata as recordings."
    )
provenance.update(
    {
        "declared_format": info.get("codebase_version"),
        "declared_episodes": info.get("total_episodes"),
        "grader_version": "0.10.0",
        "python_version": sys.version,
    }
)
(OUTPUT / "provenance.json").write_text(json.dumps(provenance, indent=2))
print(json.dumps(provenance, indent=2))

### Run the assessment
The grader produces the JSON evidence used by the guided views below. Its own CLI summary remains visible as the authoritative fallback.

**Exit 0:** configured gate passed. **Exit 1:** assessment completed, but the review/block/unknown gate failed. **Other exits:** execution error. Passing a gate is not a guarantee of training performance.


In [ ]:
# @title Run the audit (click to show code)
grader_env = dict(os.environ, NO_COLOR="1")
grader_env.pop("KALANOS_API_KEY", None)


def grade_report(suffix):
    report = OUTPUT / f"audit.{suffix}"
    command = [
        str(KALANOS),
        "grade",
        str(DATASET),
        "--report",
        str(report),
        "--fail-on",
        "blocked,review,unknown",
    ]
    result = subprocess.run(command, env=grader_env, capture_output=True, text=True)
    (OUTPUT / f"grader-{suffix}.log").write_text(result.stdout + "\n" + result.stderr)
    print(result.stdout)
    if result.returncode not in (0, 1) or not report.is_file():
        raise RuntimeError(
            f"Grading failed (exit {result.returncode}). " + result.stderr[-3000:]
        )
    print(
        "Gate:", "passed" if result.returncode == 0 else "failed; review the assessment"
    )
    return report


JSON_REPORT = grade_report("json")
summary = subprocess.run(
    [str(KALANOS), "inspect", str(JSON_REPORT)],
    capture_output=True,
    text=True,
    check=True,
)
# The full finding records (JSON lines) are kept in summary.txt and audit.json.
shown = summary.stdout.splitlines()
first_record = next((i for i, x in enumerate(shown) if x.startswith("{")), len(shown))
print("\n".join(shown[:first_record]))
(OUTPUT / "summary.txt").write_text(summary.stdout)

## 3 · Explore the evidence
The cards below are computed from **this run's JSON**, not a saved example. Unknown values remain unknown. Findings retain their metric, measured value, recorded episode key and source location; those keys are not asserted to be LeRobot source indices.

Use the metric table to see the grader's checked denominator. The evidence rows are findings, not a complete list of checks. All blocking/review findings remain available; there is no top-three cutoff.


In [ ]:
import json
import math
from html import escape


def render_guided(report):
    """Presentation only: schema 8.1, never recompute grader decisions."""

    def esc(value):
        return escape(str(value), quote=True)

    def obj(value):
        return value if isinstance(value, dict) else {}

    def number(value):
        return (
            isinstance(value, (int, float))
            and not isinstance(value, bool)
            and math.isfinite(value)
        )

    def fmt(value):
        if value is None:
            return "Not reported"
        if number(value):
            return f"{value:.4g}"
        return str(value)

    if not isinstance(report, dict) or report.get("schema_version") != "8.1.0":
        return "<p>Guided view supports schema 8.1.0. Use the CLI summary and original report for this schema; no results were inferred.</p>"
    reasons = report.get("reason_codes", [])

    def reason(value):
        if isinstance(value, int) and not isinstance(value, bool):
            if isinstance(reasons, list) and 0 <= value < len(reasons):
                return reasons[value]
            if isinstance(reasons, dict):
                return reasons.get(str(value), f"Unresolved reason reference {value}")
            return f"Unresolved reason reference {value}"
        return value if value is not None else "Not reported"

    def table(headers, rows):
        return (
            '<div style="overflow:auto"><table style="border-collapse:collapse;width:100%"><thead><tr>'
            + "".join(
                '<th style="text-align:left;padding:8px;border-bottom:2px solid #cbd5e1">'
                + esc(x)
                + "</th>"
                for x in headers
            )
            + "</tr></thead><tbody>"
            + "".join(
                "<tr>"
                + "".join(
                    '<td style="vertical-align:top;padding:8px;border-bottom:1px solid #e2e8f0">'
                    + esc(fmt(x))
                    + "</td>"
                    for x in row
                )
                + "</tr>"
                for row in rows
            )
            + "</tbody></table></div>"
        )

    score = obj(report.get("score"))
    readiness = obj(report.get("readiness"))
    counts = obj(report.get("eligibility_counts"))
    score_value = score.get("score")
    ready_value = readiness.get("score")
    cards = [
        (
            "Descriptive score",
            f"{score_value:.1f}/100" if number(score_value) else "Not reported",
        ),
        (
            "Readiness",
            f"{ready_value:.1f}/100"
            if number(ready_value)
            else "Undefined / not reported",
        ),
        ("Episodes assessed", fmt(counts.get("total"))),
        (
            "Pass / review",
            f"{fmt(counts.get('pass_count'))} / {fmt(counts.get('review'))}",
        ),
        (
            "Assessment scope",
            " · ".join(
                str(x)
                for x in (
                    obj(report.get("scope")).get("requirements_id"),
                    obj(report.get("scope")).get("policy_id"),
                )
                if x
            )
            or "Not reported",
        ),
    ]
    html = '<section style="background:#fff;color:#172554;padding:20px;border:1px solid #cbd5e1;border-radius:12px;font-family:system-ui,sans-serif"><h2>This run at a glance</h2><div style="display:flex;flex-wrap:wrap;gap:12px">'
    for title, value in cards:
        html += f'<div style="flex:1;min-width:150px;background:#f1f5f9;padding:16px;border-radius:8px"><small>{esc(title)}</small><div style="font-size:22px;font-weight:700;overflow-wrap:anywhere">{esc(value)}</div></div>'
    html += "</div><p>Descriptive score and readiness are different. Neither establishes training performance.</p>"
    html += table(
        ["Pass", "Review", "Blocked", "Unknown"],
        [[counts.get(x) for x in ("pass_count", "review", "blocked", "unknown")]],
    )
    labels = {
        "integrity.spike_pct": "sudden changes",
        "integrity.snr_db": "low signal-to-noise",
        "integrity.rotation_snr_db": "low signal-to-noise (rotation)",
    }
    review_findings = [
        x
        for x in report.get("findings") or []
        if isinstance(x, dict) and x.get("consequence") == "review"
    ]
    n_review = counts.get("review")
    if number(n_review) and n_review > 0:
        metrics_seen = sorted({str(x.get("metric_id")) for x in review_findings})
        episodes_seen = sorted({str(x.get("episode_id")) for x in review_findings})
        what = (
            ", ".join(f"{labels.get(m, m)} ({m})" for m in metrics_seen)
            or "see the evidence rows"
        )
        html += (
            f"<h3>What needs attention</h3><p><strong>{esc(n_review)} episode(s) need review.</strong> Recorded review findings: {esc(what)}"
            + (f" on {esc(len(episodes_seen))} episode(s)" if episodes_seen else "")
            + ". These are measurements tied to the evidence rows below; they do not show why the data changed. Spikes in the fixture are not shown to be the cause.</p>"
        )
    elif number(n_review):
        html += "<h3>What needs attention</h3><p>No episode was marked for review in this run. Check coverage and findings before treating that as a clean result.</p>"
    html += (
        "<p><strong>Readiness explanation:</strong> "
        + esc(
            "; ".join(str(reason(x)) for x in readiness.get("reasons", []))
            or "No explanation recorded"
        )
        + "</p>"
    )
    html += (
        '<details><summary>Recorded scope (not a capability guarantee)</summary><pre style="white-space:pre-wrap">'
        + esc(json.dumps(report.get("scope"), indent=2))
        + "</pre></details>"
    )
    metrics = obj(score.get("metrics"))
    rows = []
    for key, m in sorted(metrics.items()):
        m = obj(m)
        rows.append(
            [
                key,
                m.get("level"),
                m.get("n_graded"),
                m.get("n_flagged"),
                m.get("n_episodes_flagged"),
            ]
        )
    html += "<h3>Measured checks</h3><p>Checked and flagged counts are reported in the subject unit shown. They are not all episode counts. A missing count is not zero. Consult coverage in the CLI summary for unavailable and unrequired checks.</p>"
    html += table(
        [
            "Metric",
            "Subject unit",
            "Checked subjects",
            "Flagged subjects",
            "Affected episodes",
        ],
        rows,
    )
    findings = report.get("findings")
    if not isinstance(findings, list):
        return (
            html
            + "<p>Findings not reported. This is not evidence of no problems.</p></section>"
        )
    html += f"<h3>Investigate the evidence: {len(findings)} recorded findings</h3><p>Every finding is retained below. Open a row for its source and evidence. A severity label is not the same as a blocking decision.</p>"
    priority = {"blocked": 0, "block": 0, "review": 1}
    for f in sorted(
        (x for x in findings if isinstance(x, dict)),
        key=lambda f: (
            priority.get(f.get("consequence"), 2),
            str(f.get("metric_id")),
            str(f.get("episode_id")),
        ),
    ):
        title = " · ".join(
            str(fmt(f.get(k)))
            for k in ("consequence", "metric_id", "episode_id", "channel")
        )
        value = f"{fmt(f.get('value'))} {f.get('unit') or ''}".strip()
        html += (
            '<details style="padding:10px 0;border-top:1px solid #cbd5e1"><summary>'
            + esc(title)
            + " | "
            + esc(value)
            + "</summary>"
        )
        html += table(
            [
                "Finding ID",
                "Severity",
                "Evidence strength",
                "Source file",
                "Source field",
            ],
            [
                [
                    f.get(k)
                    for k in (
                        "id",
                        "severity",
                        "evidence_strength",
                        "source_path",
                        "source_field",
                    )
                ]
            ],
        )
        html += (
            "<p><strong>Calibration/context:</strong> "
            + esc(reason(obj(f.get("calibration")).get("reason")))
            + "</p>"
        )
        html += (
            '<pre style="white-space:pre-wrap;overflow-wrap:anywhere;background:#f8fafc;padding:10px">'
            + esc(json.dumps(f.get("evidence"), indent=2))
            + "</pre>"
        )
        html += "<p><strong>Next check:</strong> inspect this recorded episode/channel and the metric’s assumptions. If an anomaly is confirmed, investigate the source before deciding to repair, exclude, or recollect. This report does not identify the root cause.</p></details>"
    if not findings:
        html += "<p>No findings were recorded. Check coverage before interpreting that as absence of issues.</p>"
    return html + "</section>"


report = json.loads(JSON_REPORT.read_text())
display(HTML(render_guided(report)))

### Open the full report
The next cell runs a **second assessment** of the same pinned files to produce HTML. It is not a JSON conversion. Compare run metadata before comparing sampled checks. The guided cards above refer to `audit.json`; the embedded report below refers to `audit.html`.

If the report is difficult to read inline, download the HTML in the ZIP and open it in a browser. The notebook stays pinned to the validated 0.10.0 release; later report-style changes need a new package release.


In [ ]:
HTML_REPORT = grade_report("html")
# Notebook rendering may limit scripts; the downloaded HTML is the fallback.
display(HTML(HTML_REPORT.read_text()))

## 4 · Keep your results
Download the ZIP before the temporary runtime expires. It contains JSON evidence, the HTML report, CLI summary/logs, dependency versions and dataset provenance. These are local artifacts, not a new Hub publication. Review paths and diagnostic content before sharing your own private datasets' reports.


In [ ]:
BUNDLE = WORK / "kalanos-audit-results.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT.iterdir():
        if path.is_file():
            archive.write(path, path.name)
try:
    from google.colab import files
except ImportError:
    print("Outside Colab: download", BUNDLE)
    display(FileLink(str(BUNDLE)))
else:
    files.download(str(BUNDLE))

## 5 · Try a small investigation

**Exercise:** open the episodes marked for review and decide what you would check before training. Which evidence supports each check, and which would you still need to collect?

1. **Score versus decision.** Find a review episode and its measured finding. Explain why a high whole-dataset score does not cancel that review.
2. **Follow the evidence.** Pick one row; locate its episode, channel, value and calibration explanation in the downloaded JSON. A smooth/residual SNR diagnostic is not a physical sensor-noise measurement.
3. **Check what is missing.** Use the CLI coverage summary. Which capabilities were unavailable or not required? Why is that different from passing a check?
4. **Compare a control.** Change `DATASET_ID` to `kalanosai/synthetic-clean-signals`, clear `DATASET_REVISION`, then rerun from the beginning. Save both bundles. Compare only compatible scopes and coverage; do not predict the clean fixture's score without running it.

**Suggested submission:** dataset ID + revision, grader version + scope, one evidence-backed finding, one coverage limitation, and the ZIP. Do not infer a faulty sensor, bad controller, or safe training data from a metric alone.

### Continue with Kalanos
- [View the dataset](https://huggingface.co/datasets/kalanosai/synthetic-action-spikes): the public synthetic fixture used here.
- [Browse public dataset reports](https://hub.kalanos.ai): investigate existing assessments; their revisions and scopes may differ from this run.
- [Kalanos website](https://kalanos.ai): learn about the toolkit and workflows.
- [Grader source and methodology](https://github.com/KalanosAI/kalanos): run local audits and inspect decision semantics.
- [MCP integrations](https://github.com/KalanosAI/kalanos-mcp): consume published evidence in agent workflows; not required for this notebook.
- [Experiment tracking adapters](https://github.com/KalanosAI/kalanos-tracking): attach existing assessments to experiments; sharing is a separate choice.
- [Synthetic fixture generator](https://hub.kalanos.ai/synthetic-data): sign-in required.
